# Reach–avoid examples

One configurable pdSTL reach–avoid planner. A scenario lists a `workspace`, `obstacles`, `goals` (reach any one) and `visit` groups (enter one region and stay `dwell` steps); the task is

$$\mathbf{G}_{[1,H]}\,\text{safe} \;\land\; \mathbf{F}_{[1,H]}\,\text{any goal} \;\land\; \textstyle\bigwedge_{\text{groups}} \mathbf{F}_{[1,H-d]}\,\mathbf{G}_{[0,d]}\,\text{any target}.$$

The robot is a double integrator under Gaussian beliefs. Each `route` is one warm start; the planner maximizes the exact pdSTL lower bound from each and keeps the best certified plan.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from planning.runners import RESULTS_DIR, run_reach_avoid, setup_problem
from utils import load_config


def example(name):
    """Print the task, plan every route, and show the figure and animation."""
    path = ROOT / 'configs' / 'scenarios' / f'{name}.yaml'
    problem = setup_problem(load_config(path), device='cpu', with_environment=True)
    print(problem.env.get_specification(problem.cfg['H']))
    result = run_reach_avoid(str(path), save=True)
    lower, upper = result.hard_interval
    display(Markdown(f'**Exact pdSTL interval:** $[{lower:.3f}, {upper:.3f}]$'))
    display(Image(filename=str(RESULTS_DIR / f'{name}.png')))
    display(Image(filename=str(RESULTS_DIR / f'{name}.gif')))

## 1. Narrow passage (stlpy)

Reach goal A or B past four obstacles. Three routes: the 0.3 m corridor, the 0.5 m gap, and the wide left column. Centred in a corridor of width $w$, a step's collision risk is about $2\Phi(-w/2\sigma)$, so pdSTL certifies the narrow corridor lowest.

In [ ]:
example('narrow_passage')

## 2. Either–or (stlpy)

Dwell five steps in target t1 **or** t2, reach the goal, and avoid the block. The choice is one `visit` group with two regions:

```yaml
visit:
  - dwell: 5
    regions:
      t1: {x: [1, 2], y: [6, 7]}
      t2: {x: [7, 8], y: [4.5, 5.5]}
```

In [ ]:
example('either_or')